# Customer Experience Analytics for Ethiopian Fintech Apps

## Introduction

Mobile banking adoption in Ethiopia has grown rapidly in recent years, making mobile applications an essential service channel for banks. As competition among financial institutions increases, user experience has become a major factor influencing customer satisfaction, retention, and trust. Customers regularly express their opinions through Google Play Store reviews, sharing both positive experiences and frustrations related to mobile banking applications.

These reviews contain valuable business insights. Positive feedback can highlight strengths such as ease of use, fast transactions, and reliable services, while negative reviews can reveal recurring problems such as login failures, slow transfers, application crashes, OTP issues, and poor customer support. Analyzing this feedback systematically allows banks to better understand customer needs and prioritize product improvements.

The objective of this project is to build a data analytics pipeline that collects, cleans, and prepares Google Play Store reviews for downstream sentiment and thematic analysis. The cleaned dataset will later be used to identify customer satisfaction drivers, common complaints, and feature requests across multiple banking applications.

## Project Goals

The main goals of Task 1 are:

- Scrape user reviews from the Google Play Store using Python
- Collect structured review information including:
  - Review text
  - Rating
  - Review date
  - Bank name
  - Source
- Perform data quality assessment on the raw scraped data
- Clean and preprocess the dataset by:
  - Removing duplicate reviews
  - Handling missing values
  - Standardizing date formats
  - Cleaning inconsistent review text
  - Validating rating values
- Produce a clean, analysis-ready CSV dataset for later NLP and database tasks

## Target Applications

This analysis focuses on three Ethiopian banking applications available on the Google Play Store:

| Bank | Application Name | App ID |
|------|------------------|--------|
| CBE | Commercial Bank of Ethiopia Mobile Banking | `com.combanketh.mobilebanking` |
| BOA | Bank of Abyssinia Mobile Banking | `com.boa.boaMobileBanking` |
| Dashen | Dashen Super App | `com.dashen.dashensuperapp` |

The reviews collected from these applications will serve as the primary dataset for customer experience analysis throughout the project.

In [2]:
import sys
import os

# Add project root to Python path
sys.path.append(os.path.abspath(".."))

In [4]:
import pandas as pd

from src.scraper import scrape_reviews
from src.preprocess import preprocess_reviews
from src.config import BANK_APPS

In [5]:
dfs = []

for bank, app_id in BANK_APPS.items():

    print(f"Scraping {bank} reviews...")

    df = scrape_reviews(
        app_id=app_id,
        bank_name=bank,
        count=500
    )

    dfs.append(df)

Scraping CBE reviews...
Scraping BOA reviews...
Scraping Dashen reviews...


In [6]:
df_raw = pd.concat(dfs, ignore_index=True)

print(df_raw.shape)
df_raw.head()

(1500, 6)


,review_id,review,rating,date,bank,source
0,f11ba9ef-c1a1-4006-9ead-afb585624f63,Good,5,2026-05-16 19:03:11,CBE,Google Play
1,5c08f975-fcca-4b2d-8044-25490b83e988,🤙🏼🤙🏼,5,2026-05-16 15:50:50,CBE,Google Play
2,c1e25b5d-7e79-4b60-8aa5-bed69a904f62,worst,1,2026-05-16 12:15:55,CBE,Google Play
3,eb3cc438-1c10-4e72-8851-3efff6a04135,this app very full,5,2026-05-16 09:17:00,CBE,Google Play
4,f8209985-ea16-4f28-bb48-d6a7276f0f08,good apps,4,2026-05-16 07:18:33,CBE,Google Play


In [8]:
print("=" * 60)
print("RAW DATA EXPLORATION")
print("=" * 60)


print("\n1. Dataset Shape")
print("-" * 30)

print(f"Number of rows    : {df_raw.shape[0]}")
print(f"Number of columns : {df_raw.shape[1]}")


print("\n2. Dataset Preview")
print("-" * 30)

display(df_raw.head())


print("\n3. Column Data Types")
print("-" * 30)

print(df_raw.dtypes)


print("\n4. Missing Values")
print("-" * 30)

missing_values = df_raw.isnull().sum()
missing_percentage = (
    missing_values / len(df_raw) * 100
).round(2)

missing_report = pd.DataFrame({
    "missing_count": missing_values,
    "missing_percentage": missing_percentage
})

display(missing_report)


print("\n5. Duplicate Analysis")
print("-" * 30)

duplicate_ids = df_raw.duplicated(
    subset=["review_id"]
).sum()

duplicate_reviews = df_raw.duplicated(
    subset=["review"]
).sum()

print(f"Duplicate review IDs   : {duplicate_ids}")
print(f"Duplicate review texts : {duplicate_reviews}")


print("\n6. Empty Review Texts")
print("-" * 30)

empty_reviews = (
    df_raw["review"]
    .fillna("")
    .str.strip() == ""
).sum()

print(f"Empty review texts : {empty_reviews}")


print("\n7. Rating Distribution")
print("-" * 30)

rating_counts = (
    df_raw["rating"]
    .value_counts()
    .sort_index(ascending=False)
)

for rating, count in rating_counts.items():

    percentage = round(
        count / len(df_raw) * 100,
        2
    )

    bar = "█" * (count // 10)

    print(
        f"{int(rating)} Stars : "
        f"{count:>4} reviews "
        f"({percentage:>5}%) "
        f"{bar}"
    )
print("\n8. Date Exploration")
print("-" * 30)

print(f"Date dtype : {df_raw['date'].dtype}")

print("\nSample date values:")
print(df_raw["date"].head(5).to_string(index=False))


print("\n9. Reviews Per Bank")
print("-" * 30)

bank_counts = df_raw["bank"].value_counts()

for bank, count in bank_counts.items():
    print(f"{bank:<10}: {count} reviews")

print("\n" + "=" * 60)
print("RAW DATA EXPLORATION COMPLETE")
print("=" * 60)

RAW DATA EXPLORATION

1. Dataset Shape
------------------------------
Number of rows    : 1500
Number of columns : 6

2. Dataset Preview
------------------------------


,review_id,review,rating,date,bank,source
0,f11ba9ef-c1a1-4006-9ead-afb585624f63,Good,5,2026-05-16 19:03:11,CBE,Google Play
1,5c08f975-fcca-4b2d-8044-25490b83e988,🤙🏼🤙🏼,5,2026-05-16 15:50:50,CBE,Google Play
2,c1e25b5d-7e79-4b60-8aa5-bed69a904f62,worst,1,2026-05-16 12:15:55,CBE,Google Play
3,eb3cc438-1c10-4e72-8851-3efff6a04135,this app very full,5,2026-05-16 09:17:00,CBE,Google Play
4,f8209985-ea16-4f28-bb48-d6a7276f0f08,good apps,4,2026-05-16 07:18:33,CBE,Google Play



3. Column Data Types
------------------------------
review_id               str
review                  str
rating                int64
date         datetime64[us]
bank                    str
source                  str
dtype: object

4. Missing Values
------------------------------


,missing_count,missing_percentage
review_id,0,0.0
review,0,0.0
rating,0,0.0
date,0,0.0
bank,0,0.0
source,0,0.0



5. Duplicate Analysis
------------------------------
Duplicate review IDs   : 0
Duplicate review texts : 365

6. Empty Review Texts
------------------------------
Empty review texts : 0

7. Rating Distribution
------------------------------
5 Stars :  941 reviews (62.73%) ██████████████████████████████████████████████████████████████████████████████████████████████
4 Stars :  115 reviews ( 7.67%) ███████████
3 Stars :   78 reviews (  5.2%) ███████
2 Stars :   47 reviews ( 3.13%) ████
1 Stars :  319 reviews (21.27%) ███████████████████████████████

8. Date Exploration
------------------------------
Date dtype : datetime64[us]

Sample date values:
2026-05-16 19:03:11
2026-05-16 15:50:50
2026-05-16 12:15:55
2026-05-16 09:17:00
2026-05-16 07:18:33

9. Reviews Per Bank
------------------------------
CBE       : 500 reviews
BOA       : 500 reviews
Dashen    : 500 reviews

RAW DATA EXPLORATION COMPLETE


In [9]:
import os

os.makedirs("../data/raw", exist_ok=True)

df_raw.to_csv(
    "../data/raw/raw_reviews.csv",
    index=False
)

In [10]:
df_clean = preprocess_reviews(df_raw)

In [11]:
print(df_clean.shape)
print(df_clean.isnull().sum())

(1500, 6)
review_id    0
review       0
rating       0
date         0
bank         0
source       0
dtype: int64


In [ ]:
df_final = df_clean[
    ["review", "rating", "date", "bank", "source"]
].copy()

os.makedirs("../data/processed", exist_ok=True)

output_path = "../data/processed/bank_reviews_clean.csv"

df_final.to_csv(output_path, index=False)

print(f"Saved cleaned dataset to: {output_path}")




Saved cleaned dataset to: ../data/processed/bank_reviews_clean.csv


,review,rating,date,bank,source
0,Good,5,2026-05-16,CBE,Google Play
1,🤙🏼🤙🏼,5,2026-05-16,CBE,Google Play
2,worst,1,2026-05-16,CBE,Google Play
3,this app very full,5,2026-05-16,CBE,Google Play
4,good apps,4,2026-05-16,CBE,Google Play
